# QubitGuard: a first quantum error correction experiment

This notebook runs real Stim simulations using QubitGuard's engine. Install the project with `pip install -e .` in the notebook kernel's environment first. Jupyter is optional and is not needed by the web app.

An X error flips a computational basis value; a Z error flips phase. A bit-flip repetition code encodes a logical value across an odd number of data qubits and checks neighboring Z parities. It protects one error type. A surface code uses both X- and Z-type stabilizers to protect arbitrary Pauli errors. Neither encoding means copying an arbitrary unknown quantum state.

We begin with the repetition model: independent data X errors and noisy parity measurements each round, ideal parity gates and ideal final data readout. The full assumptions are in `docs/science.md`.

In [1]:
import numpy as np
from math import comb
from qubitguard.engine import build_circuit, majority_decode, run_experiment
from qubitguard.models import ExperimentConfig

circuit = build_circuit('repetition', distance=3, rounds=3, p=0.05)
print(f'{circuit.num_qubits} physical qubits, {circuit.num_detectors} detectors, {circuit.num_observables} observable')
print(circuit)


5 physical qubits, 8 detectors, 1 observable
R 0 1 2 3 4
X_ERROR(0.05) 0 1 2
CX 0 3 1 3 1 4 2 4
X_ERROR(0.05) 3 4
MR 3 4
DETECTOR(0.5, 0, 0) rec[-2]
DETECTOR(1.5, 0, 0) rec[-1]
TICK
X_ERROR(0.05) 0 1 2
CX 0 3 1 3 1 4 2 4
X_ERROR(0.05) 3 4
MR 3 4
DETECTOR(0.5, 0, 1) rec[-2] rec[-4]
DETECTOR(1.5, 0, 1) rec[-1] rec[-3]
TICK
X_ERROR(0.05) 0 1 2
CX 0 3 1 3 1 4 2 4
X_ERROR(0.05) 3 4
MR 3 4
DETECTOR(0.5, 0, 2) rec[-2] rec[-4]
DETECTOR(1.5, 0, 2) rec[-1] rec[-3]
TICK
M 0 1 2
DETECTOR(0.5, 0, 3) rec[-3] rec[-2] rec[-5]
DETECTOR(1.5, 0, 3) rec[-2] rec[-1] rec[-4]
OBSERVABLE_INCLUDE(0) rec[-1]


## Detection events and logical observables

A detector is an expected parity relation. A detection event occurs when that relation is violated. Comparing successive noisy checks helps distinguish changes over time. A detector is evidence of an error, not a unique physical error location. The final detector layer closes the memory experiment using ideal data readout.

The decoder receives only the detection events. The sampled observable is used afterward to score recovery. It is not an input to decoding.

In [2]:
syndrome, actual = circuit.compile_detector_sampler(seed=42).sample(12, separate_observables=True)
predicted = majority_decode(syndrome, distance=3, rounds=3)
print('shot | detection event indices | actual flip | predicted flip | recovered')
for i in range(12):
    print(i, np.flatnonzero(syndrome[i]).tolist(), int(actual[i, 0]), int(predicted[i, 0]), bool(actual[i, 0] == predicted[i, 0]), sep=' | ')


shot | detection event indices | actual flip | predicted flip | recovered
0 | [3] | 1 | 1 | True
1 | [3, 5] | 0 | 0 | True
2 | [] | 0 | 0 | True
3 | [2, 4] | 0 | 0 | True
4 | [4, 5] | 0 | 0 | True
5 | [] | 0 | 0 | True
6 | [1, 3, 4, 6] | 0 | 0 | True
7 | [0] | 0 | 0 | True
8 | [3] | 0 | 1 | False
9 | [] | 0 | 0 | True
10 | [2, 4] | 0 | 0 | True
11 | [0, 1, 4, 6] | 0 | 0 | True


## Benchmark identical shots with two decoders

Majority vote reconstructs final bit patterns from parity checks and selects the lower-weight pattern. Its temporal XOR cancels intermediate readout noise, but discards useful history. MWPM uses a weighted detector graph to explain the entire event history.

Logical error rate means wrong logical predictions divided by shots for the whole memory experiment. Wilson confidence intervals quantify finite-sampling uncertainty; zero failures is not proof of a zero underlying failure probability.

In [3]:
config = ExperimentConfig(distances=[3, 5, 7], physical_error_rates=[0.01, 0.03, 0.05], rounds=5, shots=10000, seed=42)
result = run_experiment(config)
print('distance | physical p | decoder | failures / shots | logical rate | 95% CI')
for cell in result['results']:
    for decoder in cell['decoders']:
        lo, hi = decoder['confidence_interval']
        print(f"{cell['distance']:8d} | {cell['physical_error_rate']:.2f} | {decoder['decoder']:8s} | {decoder['failures']:4d} / {cell['shots']} | {decoder['logical_error_rate']:.5f} | [{lo:.5f}, {hi:.5f}]")


distance | physical p | decoder | failures / shots | logical rate | 95% CI
       3 | 0.01 | mwpm     |   48 / 10000 | 0.00480 | [0.00362, 0.00636]
       3 | 0.01 | majority |   80 / 10000 | 0.00800 | [0.00643, 0.00994]
       3 | 0.03 | mwpm     |  360 / 10000 | 0.03600 | [0.03252, 0.03983]
       3 | 0.03 | majority |  491 / 10000 | 0.04910 | [0.04504, 0.05351]
       3 | 0.05 | mwpm     |  892 / 10000 | 0.08920 | [0.08377, 0.09495]
       3 | 0.05 | majority | 1076 / 10000 | 0.10760 | [0.10168, 0.11382]
       5 | 0.01 | mwpm     |    3 / 10000 | 0.00030 | [0.00010, 0.00088]
       5 | 0.01 | majority |    8 / 10000 | 0.00080 | [0.00041, 0.00158]
       5 | 0.03 | mwpm     |   65 / 10000 | 0.00650 | [0.00510, 0.00828]
       5 | 0.03 | majority |  204 / 10000 | 0.02040 | [0.01781, 0.02336]
       5 | 0.05 | mwpm     |  274 / 10000 | 0.02740 | [0.02438, 0.03079]
       5 | 0.05 | majority |  638 / 10000 | 0.06380 | [0.05918, 0.06876]
       7 | 0.01 | mwpm     |    0 / 10000 | 0.000

## An independent analytic check

After `r` rounds, independent data X errors accumulate to an odd parity with probability `q = (1 - (1 - 2p)^r) / 2`. For odd distance `d`, majority fails when more than half of the data bits have flipped. This gives a binomial tail that we can compare to the simulator. This expression applies to this ideal-final-readout baseline, not the surface-code MWPM experiment.

In [4]:
p, r = 0.05, config.rounds
q = (1 - (1 - 2*p)**r) / 2
for cell in result['results']:
    if cell['physical_error_rate'] != p:
        continue
    d = cell['distance']
    expected = sum(comb(d, k) * q**k * (1-q)**(d-k) for k in range(d//2 + 1, d + 1))
    observed = next(x for x in cell['decoders'] if x['decoder'] == 'majority')
    print(f"d={d}: analytic={expected:.5f}, sampled={observed['logical_error_rate']:.5f}, CI={observed['confidence_interval']}")
print('Versions:', result['provenance']['versions'])


d=3: analytic=0.10861, sampled=0.10760, CI=[0.10167654822744618, 0.1138248136938608]
d=5: analytic=0.06164, sampled=0.06380, CI=[0.059175411817862796, 0.06875958836071547]
d=7: analytic=0.03614, sampled=0.03630, CI=[0.032808611118288936, 0.040147508970676524]
Versions: {'stim': '1.16.0', 'pymatching': '2.4.0', 'numpy': '2.4.6'}


## Next experiments

1. Increase shots when confidence intervals are wide. Record root seed, versions and batch size.
2. Add the `identity` decoder to measure the no-correction baseline.
3. Try `code='surface_z'` with `decoders=['mwpm', 'identity']` at p=0.001 and p=0.005. Its circuit-level model differs from the repetition model.
4. Compare fixed rounds with rounds proportional to distance in separate runs. Explain the changed experiment duration.

Distance suppression under this model is useful evidence, but a finite sweep does not establish a hardware threshold. Batch timing is throughput on the host CPU, not real-time hardware decoder latency.

Primary references: [Stim](https://github.com/quantumlib/Stim), [PyMatching](https://pymatching.readthedocs.io/en/stable/), and [Deltakit](https://github.com/Deltakit/deltakit).